# Kienzle Cutting-Force Model — Bayesian Parameter Estimation (MCMC)
**Author:** Thomas M. Rudolf  
**Course:** Métodos Analíticos — Maestría en Ciencia de Datos, ITAM  

## Overview

This notebook estimates the Kienzle cutting-force model parameters
$k_{c1.1}$ and $m_c$ from CNC spindle motor-current (torque) signals
using Bayesian Markov-Chain Monte Carlo (MCMC) in **Stan** (via CmdStanPy)
and **JAGS** (via PyJAGS).  All physical modelling re-uses the pre-built
Python classes `KienzleModel` and `MillingSignalUtils`.

### Physical model
$$
F_{c,i} = k_{c1.1}\; a_p\; f_z^{1-m_c}\; \sin(\kappa)^{m_c}\; \sin(\varphi_{\text{eff},i})^{1-m_c}
$$

$$
T_c = r_{\text{tool}} \sum_{i=1}^{z} F_{c,i}
$$

### Notebook structure

| Section | Content |
|---------|---------|
| 1 | Setup & imports |
| 2 | Prior distributions |
| 3 | Generative model — synthetic data |
| 4 | Sensitivity analysis ($m_c$, $k_{c1.1}$, $\Delta\varphi$) |
| 5 | Grid-based posterior check |
| 6 | Stan MCMC — clean signal |
| 7 | Stan MCMC — noisy signal |
| 8 | JAGS MCMC — max-torque-per-revolution |
| 9 | Stan vs JAGS comparison |
| 10 | Inferential calibration ($z$-score, posterior concentration) |
| 11 | Conclusions |


## 1  Setup & Imports

In [ ]:
# ── standard library ────────────────────────────────────────────
import sys, os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import beta as beta_dist, norm as norm_dist

# ── project modules (must be in the same directory as this notebook) ──
sys.path.insert(0, os.path.abspath("."))
from kienzle_model import KienzleModel
from kienzle_utils import MillingSignalUtils as MSU

print("KienzleModel  :", KienzleModel.__module__)
print("MillingSignalUtils:", MSU.__module__)


In [ ]:
# ── CmdStanPy (Stan) ─────────────────────────────────────────────
try:
    import cmdstanpy
    from cmdstanpy import CmdStanModel
    STAN_AVAILABLE = True
    print(f"CmdStanPy {cmdstanpy.__version__} available")
except ImportError:
    STAN_AVAILABLE = False
    print("CmdStanPy not installed — Stan cells will be skipped.")

# ── PyJAGS ───────────────────────────────────────────────────────
try:
    import pyjags
    JAGS_AVAILABLE = True
    print("PyJAGS available")
except ImportError:
    JAGS_AVAILABLE = False
    print("PyJAGS not installed — JAGS cells will be skipped.")

# ── ArviZ (diagnostics / plots) ──────────────────────────────────
try:
    import arviz as az
    ARVIZ_AVAILABLE = True
    print(f"ArviZ {az.__version__} available")
except ImportError:
    ARVIZ_AVAILABLE = False
    print("ArviZ not installed — some diagnostic plots will be skipped.")


In [ ]:
# ── Global plot style ────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":      130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid":       True,
    "grid.alpha":      0.35,
    "font.size":       11,
})
COLORS = {"clean": "#1f77b4", "noisy": "#ff7f0e",
          "prior": "#aec7e8", "posterior": "#d62728",
          "stan":  "#2ca02c", "jags":  "#9467bd"}


## 2  Prior Distributions

Material data from [machiningdoctor.com](https://www.machiningdoctor.com/glossary/specific-cutting-force-kc-kc1/)
(metals only, $k_{c1.1} > 1000$ MPa).

| Parameter | Distribution | Rationale |
|-----------|-------------|-----------|
| $m_c$ | Beta($\alpha$, $\beta$) | Bounded (0, 1); moment-matched from DB statistics |
| $k_{c1.1}$ | Normal($\mu$, $\sigma^2$) | Weakly informative; DB mean ± SD |
| $\Delta\varphi$ | Uniform(0, $2\pi$) | Unknown initial phase |


In [ ]:
# ── Material database values (steel grades; kc11 in MPa) ─────────
# Representative values from the DB used in the QMD files
mc_db   = np.array([0.17, 0.21, 0.22, 0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.30])
kc11_db = np.array([1350, 1500, 1600, 1700, 1775, 1800, 1850, 1900, 2000, 2100])  # MPa

mean_mc   = float(np.mean(mc_db))
sd_mc     = float(np.std(mc_db, ddof=1))
mean_kc11 = float(np.mean(kc11_db)) * 1e6   # Pa
sd_kc11   = float(np.std(kc11_db, ddof=1)) * 1e6

alpha_mc, beta_mc = MSU.beta_dist_param(mean=mean_mc, var=sd_mc**2)

print(f"mc    : mean={mean_mc:.4f}  sd={sd_mc:.4f}")
print(f"       Beta alpha={alpha_mc:.2f}  beta={beta_mc:.2f}")
print(f"kc11  : mean={mean_kc11/1e6:.1f} MPa  sd={sd_kc11/1e6:.1f} MPa")


In [ ]:
# ── Visualise prior distributions ────────────────────────────────
mc_grid   = np.linspace(0.01, 0.99, 500)
kc11_grid = np.linspace(500e6, 4000e6, 500)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Beta prior for mc
axes[0].plot(mc_grid,
             beta_dist.pdf(mc_grid, alpha_mc, beta_mc),
             color=COLORS["prior"], lw=2, label=f"Beta({alpha_mc:.1f}, {beta_mc:.1f})")
axes[0].axvline(mean_mc, color="red", ls="--", label=f"mean={mean_mc:.3f}")
axes[0].set_xlabel(r"$m_c$")
axes[0].set_ylabel("Density")
axes[0].set_title(r"Prior: $m_c$")
axes[0].legend()

# Normal prior for kc11
axes[1].plot(kc11_grid / 1e6,
             norm_dist.pdf(kc11_grid, mean_kc11, sd_kc11),
             color=COLORS["prior"], lw=2,
             label=f"N({mean_kc11/1e6:.0f}, {sd_kc11/1e6:.0f}) MPa")
axes[1].axvline(mean_kc11 / 1e6, color="red", ls="--",
                label=f"mean={mean_kc11/1e6:.0f} MPa")
axes[1].set_xlabel(r"$k_{c1.1}$ / MPa")
axes[1].set_ylabel("Density")
axes[1].set_title(r"Prior: $k_{c1.1}$")
axes[1].legend()

fig.tight_layout()
plt.show()


## 3  Generative Model — Synthetic Data

Values for **ST45** (the experimental material) are derived from similar
steel grades ST32 / ST52 in the database.  The `KienzleModel` class handles
all physics; `MillingSignalUtils` provides signal-processing helpers.


In [ ]:
# ── Process parameters ───────────────────────────────────────────
np.random.seed(42)

Z       = 4            # number of flutes
AP      = 2.0          # axial depth [mm]
FZ      = 0.15         # feed / tooth [mm]
KAPPA   = 13.0         # cutting-edge angle [°]
R_TOOL  = 40.0         # tool radius [mm]
PHI_IN  = 0.0          # entry angle [rad]
PHI_OUT = np.radians(120.0)  # exit  angle [rad]
OMEGA   = 600.0        # spindle speed [rpm]

# ST45 material priors (from ST32, ST52 in DB)
kc11_true = np.mean([1350, 1775, 1775]) * 1e6   # Pa → use as N/mm² below
mc_true   = np.mean([0.21, 0.24, 0.24])
sd_mc_st45   = float(np.std([0.21, 0.24, 0.24], ddof=1))
sd_kc11_st45 = float(np.std([1350, 1775, 1775], ddof=1)) * 1e6
alpha_mc_st45, beta_mc_st45 = MSU.beta_dist_param(mc_true, sd_mc_st45**2)

N_SIM = 2880   # number of angular samples (≈ 20 revolutions × 144 pts/rev)
t_sim = np.linspace(0, N_SIM / (OMEGA / 60 * 144), N_SIM)

# Instantiate model (kc11 in N/mm²)
model_clean = KienzleModel(
    mc=mc_true, kc11=kc11_true/1e6,
    phi_in=PHI_IN, phi_out=PHI_OUT,
    fz=FZ, ap=AP, omega=OMEGA, time=t_sim,
    z=Z, r_tool=R_TOOL, kappa_deg=KAPPA,
    white_noise_amplitude=0.0, normal_noise_std=0.0,
    seed=42,
)

model_noisy = KienzleModel(
    mc=mc_true, kc11=kc11_true/1e6,
    phi_in=PHI_IN, phi_out=PHI_OUT,
    fz=FZ, ap=AP, omega=OMEGA, time=t_sim,
    z=Z, r_tool=R_TOOL, kappa_deg=KAPPA,
    normal_noise_std=2.0,   # N·mm
    seed=42,
)

df_sim = model_clean.to_dataframe()
df_sim["Mc_noisy"] = model_noisy.torque_time_series(add_noise=True)

print(model_clean)
print(f"\nSNR (noisy model): {model_noisy.snr_db():.1f} dB")


In [ ]:
# ── Plot one revolution of the torque signal ─────────────────────
phi_vec = df_sim["phi"].values
idx_rev = phi_vec < 2 * np.pi   # first full revolution

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Individual tooth contributions (first revolution)
phi_deg = np.degrees(phi_vec[idx_rev])
for i in range(Z):
    Fci_i = np.array([
        model_clean._Fci_at_phi(p)[i] for p in phi_vec[idx_rev]
    ])
    axes[0].plot(phi_deg, Fci_i, label=f"Tooth {i+1}", alpha=0.85)
axes[0].set_xlabel(r"$\varphi$ / °")
axes[0].set_ylabel(r"$F_{c,i}$ / N")
axes[0].set_title("Individual cutting forces (one revolution)")
axes[0].legend(fontsize=9)
axes[0].axvline(360, color="k", ls="--", alpha=0.4)

# Clean vs noisy torque
axes[1].plot(phi_deg, df_sim["Mc_clean"].values[idx_rev],
             color=COLORS["clean"], lw=1.5, label="Clean $T_c$")
axes[1].plot(phi_deg, df_sim["Mc_noisy"].values[idx_rev],
             color=COLORS["noisy"], lw=0.8, alpha=0.7, label="Noisy $T_c$")
axes[1].axvline(360, color="k", ls="--", alpha=0.4)
axes[1].set_xlabel(r"$\varphi$ / °")
axes[1].set_ylabel(r"$T_c$ / N·mm")
axes[1].set_title("Cutting torque — clean vs noisy")
axes[1].legend()

fig.tight_layout()
plt.show()


## 4  Sensitivity Analysis

How does $T_c$ change when $m_c$, $k_{c1.1}$, or $\Delta\varphi$ are varied?


In [ ]:
# ── 4a  Vary mc (0.15 → 0.40, kc11 fixed at mean) ───────────────
mc_vals = np.linspace(0.15, 0.40, 12)
t_short = np.linspace(0, t_sim[N_SIM // 20], N_SIM // 20)  # ~1 revolution

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True, sharey=False)
axes = axes.ravel()

for k, mc_k in enumerate(mc_vals):
    m = KienzleModel(
        mc=mc_k, kc11=kc11_true/1e6,
        phi_in=PHI_IN, phi_out=PHI_OUT,
        fz=FZ, ap=AP, omega=OMEGA, time=t_short,
        z=Z, r_tool=R_TOOL, kappa_deg=KAPPA,
        normal_noise_std=0.0, seed=0,
    )
    Tc = m.torque_time_series(add_noise=False)
    phi_k = m.phi_time_series()
    axes[k].plot(np.degrees(phi_k), Tc, lw=1.2)
    axes[k].set_title(f"$m_c$ = {mc_k:.3f}", fontsize=9)
    axes[k].set_xlabel(r"$\varphi$/°", fontsize=8)
    axes[k].set_ylabel(r"$T_c$", fontsize=8)

fig.suptitle(r"Sensitivity: varying $m_c$ ($k_{{c1.1}}$ fixed)", fontsize=12)
fig.tight_layout()
plt.show()


In [ ]:
# ── 4b  Vary kc11 (1000 → 4000 MPa, mc fixed at mean) ───────────
kc11_vals = np.linspace(1000e6, 4000e6, 12)

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True, sharey=False)
axes = axes.ravel()

for k, kc_k in enumerate(kc11_vals):
    m = KienzleModel(
        mc=mc_true, kc11=kc_k/1e6,
        phi_in=PHI_IN, phi_out=PHI_OUT,
        fz=FZ, ap=AP, omega=OMEGA, time=t_short,
        z=Z, r_tool=R_TOOL, kappa_deg=KAPPA,
        normal_noise_std=0.0, seed=0,
    )
    Tc = m.torque_time_series(add_noise=False)
    axes[k].plot(np.degrees(m.phi_time_series()), Tc, color=COLORS["jags"], lw=1.2)
    axes[k].set_title(f"$k_{{c1.1}}$ = {kc_k/1e6:.0f} MPa", fontsize=9)
    axes[k].set_xlabel(r"$\varphi$/°", fontsize=8)
    axes[k].set_ylabel(r"$T_c$", fontsize=8)

fig.suptitle(r"Sensitivity: varying $k_{{c1.1}}$ ($m_c$ fixed)", fontsize=12)
fig.tight_layout()
plt.show()


In [ ]:
# ── 4c  Vary angular offset dphi (phase only) ────────────────────
dphi_vals = np.linspace(0, 2 * np.pi, 12, endpoint=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True, sharey=False)
axes = axes.ravel()

for k, dphi_k in enumerate(dphi_vals):
    m = KienzleModel(
        mc=mc_true, kc11=kc11_true/1e6,
        phi_in=PHI_IN, phi_out=PHI_OUT,
        fz=FZ, ap=AP, omega=OMEGA, time=t_short,
        z=Z, r_tool=R_TOOL, kappa_deg=KAPPA,
        phi0_deg=np.degrees(dphi_k),
        normal_noise_std=0.0, seed=0,
    )
    Tc = m.torque_time_series(add_noise=False)
    axes[k].plot(np.degrees(m.phi_time_series()), Tc, color=COLORS["noisy"], lw=1.2)
    axes[k].set_title(f"$\\Delta\\varphi$ = {np.degrees(dphi_k):.0f}°", fontsize=9)
    axes[k].set_xlabel(r"$\varphi$/°", fontsize=8)
    axes[k].set_ylabel(r"$T_c$", fontsize=8)

fig.suptitle(r"Sensitivity: varying angular offset $\Delta\varphi$ (phase shift only)",
             fontsize=12)
fig.tight_layout()
plt.show()


## 5  Grid-Based Posterior Check

Before committing to full MCMC, a 1-D grid search confirms that the
likelihood surface has a clear peak near the true parameters.

$$
p(m_c \mid T_c) \propto p(T_c \mid m_c)\, p(m_c)
$$


In [ ]:
# ── 5a  Posterior for mc (kc11 fixed) ───────────────────────────
# Use a small dataset (20 angular points) for speed
N_GRID = 20
t_grid = np.linspace(0, t_sim[N_GRID - 1], N_GRID)

mc_obs = 0.24          # "observed" true value
m_obs  = KienzleModel(
    mc=mc_obs, kc11=kc11_true/1e6,
    phi_in=PHI_IN, phi_out=PHI_OUT,
    fz=FZ, ap=AP, omega=OMEGA, time=t_grid,
    z=Z, r_tool=R_TOOL, kappa_deg=KAPPA,
    normal_noise_std=0.0, seed=0,
)
Tc_obs_grid = m_obs.torque_time_series(add_noise=False)
phi_obs_grid = m_obs.phi_time_series()

sigma_grid = 50.0  # N·mm observation noise assumed for grid

mc_search = np.linspace(0.10, 0.45, 500)
log_post  = np.zeros(len(mc_search))
log_prior = beta_dist.logpdf(mc_search, alpha_mc_st45, beta_mc_st45)

for j, mc_j in enumerate(mc_search):
    m_j = KienzleModel(
        mc=mc_j, kc11=kc11_true/1e6,
        phi_in=PHI_IN, phi_out=PHI_OUT,
        fz=FZ, ap=AP, omega=OMEGA, time=t_grid,
        z=Z, r_tool=R_TOOL, kappa_deg=KAPPA,
        normal_noise_std=0.0, seed=0,
    )
    Tc_pred = m_j.torque_time_series(phi_ext=phi_obs_grid, add_noise=False)
    log_lik  = np.sum(norm_dist.logpdf(Tc_obs_grid, loc=Tc_pred, scale=sigma_grid))
    log_post[j] = log_lik + log_prior[j]

post_mc = np.exp(log_post - log_post.max())
post_mc /= post_mc.sum() * (mc_search[1] - mc_search[0])
prior_mc = beta_dist.pdf(mc_search, alpha_mc_st45, beta_mc_st45)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mc_search, prior_mc / prior_mc.max(), color=COLORS["prior"],
        lw=2, label="Prior (normalised)")
ax.plot(mc_search, post_mc / post_mc.max(), color=COLORS["posterior"],
        lw=2, label="Posterior (normalised)")
ax.axvline(mc_obs,  color="green", ls="--", label=f"True $m_c$ = {mc_obs}")
ax.axvline(mc_search[np.argmax(post_mc)], color="red", ls=":",
           label=f"MAP = {mc_search[np.argmax(post_mc)]:.3f}")
ax.set_xlabel(r"$m_c$")
ax.set_ylabel("Normalised density")
ax.set_title(r"Grid posterior: $p(m_c \mid T_c)$")
ax.legend()
plt.tight_layout()
plt.show()

print(f"MAP estimate mc = {mc_search[np.argmax(post_mc)]:.4f}  (true = {mc_obs})")


In [ ]:
# ── 5b  Posterior for kc11 (mc fixed) ───────────────────────────
kc11_obs = 1700e6  # N/mm²·1e6 = Pa; model uses N/mm²
m_kc_obs = KienzleModel(
    mc=mc_true, kc11=kc11_obs/1e6,
    phi_in=PHI_IN, phi_out=PHI_OUT,
    fz=FZ, ap=AP, omega=OMEGA, time=t_grid,
    z=Z, r_tool=R_TOOL, kappa_deg=KAPPA,
    normal_noise_std=0.0, seed=0,
)
Tc_obs_kc = m_kc_obs.torque_time_series(add_noise=False)
phi_obs_kc = m_kc_obs.phi_time_series()

kc11_search = np.linspace(500e6, 4000e6, 500)  # Pa
log_post_kc = np.zeros(len(kc11_search))
log_prior_kc = norm_dist.logpdf(kc11_search, mean_kc11, sd_kc11)

for j, kc_j in enumerate(kc11_search):
    m_j = KienzleModel(
        mc=mc_true, kc11=kc_j/1e6,
        phi_in=PHI_IN, phi_out=PHI_OUT,
        fz=FZ, ap=AP, omega=OMEGA, time=t_grid,
        z=Z, r_tool=R_TOOL, kappa_deg=KAPPA,
        normal_noise_std=0.0, seed=0,
    )
    Tc_pred = m_j.torque_time_series(phi_ext=phi_obs_kc, add_noise=False)
    log_lik = np.sum(norm_dist.logpdf(Tc_obs_kc, loc=Tc_pred, scale=sigma_grid))
    log_post_kc[j] = log_lik + log_prior_kc[j]

post_kc = np.exp(log_post_kc - log_post_kc.max())
post_kc /= post_kc.sum() * (kc11_search[1] - kc11_search[0])
prior_kc = norm_dist.pdf(kc11_search, mean_kc11, sd_kc11)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(kc11_search / 1e6, prior_kc / prior_kc.max(),
        color=COLORS["prior"], lw=2, label="Prior (normalised)")
ax.plot(kc11_search / 1e6, post_kc / post_kc.max(),
        color=COLORS["posterior"], lw=2, label="Posterior (normalised)")
ax.axvline(kc11_obs / 1e6, color="green", ls="--",
           label=f"True $k_{{c1.1}}$ = {kc11_obs/1e6:.0f} MPa")
ax.axvline(kc11_search[np.argmax(post_kc)] / 1e6, color="red", ls=":",
           label=f"MAP = {kc11_search[np.argmax(post_kc)]/1e6:.0f} MPa")
ax.set_xlabel(r"$k_{c1.1}$ / MPa")
ax.set_ylabel("Normalised density")
ax.set_title(r"Grid posterior: $p(k_{{c1.1}} \mid T_c)$")
ax.legend()
plt.tight_layout()
plt.show()


## 6  Stan MCMC — Clean Signal

### Stan model (STAN_kienzle_cont.stan)

The Stan model marginalises over the continuous torque signal. 
$m_c \sim \text{Beta}(\alpha, \beta)$ and $k_{c1.1} \sim \mathcal{N}(\mu_{kc}, \sigma_{kc})$.

```stan
// STAN_kienzle_cont.stan
data {
  int<lower=1> k;
  vector[k] Tc;
  vector[k] phi;
  real<lower=0> ap;
  real<lower=0> fz;
  int<lower=1> z;
  real<lower=0> rtool;
  real<lower=0> kappa;
  real m_kc;
  real<lower=0> sd_kc;
  real<lower=0> alpha_mc;
  real<lower=0> beta_mc;
  real phi_ent;
  real phi_exit;
}
parameters {
  real<lower=0,upper=1> mc;
  real<lower=0> kc11;
  real<lower=0> sigma;
}
model {
  mc   ~ beta(alpha_mc, beta_mc);
  kc11 ~ normal(m_kc, sd_kc);
  sigma ~ exponential(1.0 / 10.0);
  for (n in 1:k) {
    real Tc_pred = 0;
    for (i in 1:z) {
      real phi_eff = fmod(phi[n] + (i-1) * 2*pi() / z, 2*pi());
      if (phi_eff >= phi_ent && phi_eff <= phi_exit)
        Tc_pred += kc11 * ap * fz^(1-mc) * sin(kappa)^mc
                   * sin(phi_eff)^(1-mc) * rtool;
    }
    Tc[n] ~ normal(Tc_pred, sigma);
  }
}
```


In [ ]:
# ── Write Stan model to disk ────────────────────────────────────
STAN_MODEL_CODE = """
data {
  int<lower=1> k;
  vector[k] Tc;
  vector[k] phi;
  real<lower=0> ap;
  real<lower=0> fz;
  int<lower=1> z;
  real<lower=0> rtool;
  real<lower=0> kappa;
  real m_kc;
  real<lower=0> sd_kc;
  real<lower=0> alpha_mc;
  real<lower=0> beta_mc;
  real phi_ent;
  real phi_exit;
}
parameters {
  real<lower=0,upper=1> mc;
  real<lower=0> kc11;
  real<lower=0> sigma;
}
model {
  mc    ~ beta(alpha_mc, beta_mc);
  kc11  ~ normal(m_kc, sd_kc);
  sigma ~ exponential(0.1);
  for (n in 1:k) {
    real Tc_pred = 0;
    for (i in 1:z) {
      real phi_eff = fmod(phi[n] + (i - 1) * 2.0 * pi() / z, 2.0 * pi());
      if (phi_eff >= phi_ent && phi_eff <= phi_exit)
        Tc_pred += kc11 * ap * pow(fz, 1 - mc) * pow(sin(kappa), mc)
                   * pow(sin(phi_eff), 1 - mc) * rtool;
    }
    Tc[n] ~ normal(Tc_pred, sigma);
  }
}
generated quantities {
  vector[k] log_lik;
  for (n in 1:k) {
    real Tc_pred = 0;
    for (i in 1:z) {
      real phi_eff = fmod(phi[n] + (i - 1) * 2.0 * pi() / z, 2.0 * pi());
      if (phi_eff >= phi_ent && phi_eff <= phi_exit)
        Tc_pred += kc11 * ap * pow(fz, 1 - mc) * pow(sin(kappa), mc)
                   * pow(sin(phi_eff), 1 - mc) * rtool;
    }
    log_lik[n] = normal_lpdf(Tc[n] | Tc_pred, sigma);
  }
}
"""

stan_model_path = "STAN_kienzle_cont.stan"
with open(stan_model_path, "w") as f:
    f.write(STAN_MODEL_CODE)
print(f"Stan model written to: {stan_model_path}")


In [ ]:
# ── Build Stan data dict (clean signal) ─────────────────────────
# Use the model's to_stan_data() method
STAN_data_clean = model_clean.to_stan_data(
    Mc_observed = df_sim["Mc_clean"].values,
    phi_observed= df_sim["phi"].values,
    m_kc        = mean_kc11 / 1e6,   # N/mm²  (Stan model is in N/mm²)
    sd_kc       = sd_kc11_st45 / 1e6,
    alpha_mc    = alpha_mc_st45,
    beta_mc     = beta_mc_st45,
)
# Add entry and exit angles required by this Stan model
STAN_data_clean["phi_ent"]  = float(PHI_IN)
STAN_data_clean["phi_exit"] = float(PHI_OUT)

print("Stan data keys :", list(STAN_data_clean.keys()))
print(f"k = {STAN_data_clean['k']}  m_kc = {STAN_data_clean['m_kc']:.1f} N/mm²  "
      f"alpha_mc = {STAN_data_clean['alpha_mc']:.2f}")


In [ ]:
# ── Run Stan (clean signal) ──────────────────────────────────────
if STAN_AVAILABLE:
    stan_model = CmdStanModel(stan_file=stan_model_path)
    fit_stan_clean = stan_model.sample(
        data           = STAN_data_clean,
        chains         = 4,
        parallel_chains= 4,
        iter_warmup    = 1000,
        iter_sampling  = 2000,
        step_size      = 0.1,
        show_progress  = False,
        show_console   = False,
    )
    print("Stan (clean) — summary:")
    print(fit_stan_clean.summary(variables=["mc", "kc11", "sigma"]))
else:
    print("Stan not available — skipping.")
    fit_stan_clean = None


In [ ]:
# ── Prior vs Posterior plot (clean) ─────────────────────────────
if fit_stan_clean is not None:
    import pandas as pd
    draws_mc_clean   = fit_stan_clean.stan_variable("mc")
    draws_kc11_clean = fit_stan_clean.stan_variable("kc11")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # mc
    mc_x = np.linspace(0.01, 0.99, 300)
    axes[0].fill_between(mc_x,
                         beta_dist.pdf(mc_x, alpha_mc_st45, beta_mc_st45),
                         alpha=0.4, color=COLORS["prior"], label="Prior")
    axes[0].hist(draws_mc_clean, bins=60, density=True,
                 color=COLORS["posterior"], alpha=0.65, label="Posterior")
    axes[0].axvline(mc_true, color="green", ls="--", lw=2,
                    label=f"True = {mc_true:.3f}")
    axes[0].set_xlabel(r"$m_c$")
    axes[0].set_title(r"Stan (clean): $m_c$")
    axes[0].legend(fontsize=9)

    # kc11
    kc_x = np.linspace(500, 4000, 300)  # MPa
    axes[1].fill_between(kc_x,
                         norm_dist.pdf(kc_x * 1e6, mean_kc11, sd_kc11_st45),
                         alpha=0.4, color=COLORS["prior"], label="Prior")
    axes[1].hist(draws_kc11_clean, bins=60, density=True,
                 color=COLORS["posterior"], alpha=0.65, label="Posterior")
    axes[1].axvline(kc11_true / 1e6, color="green", ls="--", lw=2,
                    label=f"True = {kc11_true/1e6:.0f} MPa")
    axes[1].set_xlabel(r"$k_{c1.1}$ / N·mm$^{-2}$")
    axes[1].set_title(r"Stan (clean): $k_{{c1.1}}$")
    axes[1].legend(fontsize=9)

    fig.suptitle("Stan MCMC — Prior vs Posterior (clean signal)", fontsize=12)
    fig.tight_layout()
    plt.show()


## 7  Stan MCMC — Noisy Signal

In [ ]:
# ── Build Stan data (noisy signal) ───────────────────────────────
STAN_data_noisy = model_noisy.to_stan_data(
    Mc_observed = df_sim["Mc_noisy"].values,
    phi_observed= df_sim["phi"].values,
    m_kc        = mean_kc11 / 1e6,
    sd_kc       = sd_kc11_st45 / 1e6,
    alpha_mc    = alpha_mc_st45,
    beta_mc     = beta_mc_st45,
)
STAN_data_noisy["phi_ent"]  = float(PHI_IN)
STAN_data_noisy["phi_exit"] = float(PHI_OUT)

if STAN_AVAILABLE:
    fit_stan_noisy = stan_model.sample(
        data           = STAN_data_noisy,
        chains         = 4,
        parallel_chains= 4,
        iter_warmup    = 1000,
        iter_sampling  = 2000,
        step_size      = 0.1,
        show_progress  = False,
        show_console   = False,
    )
    print("Stan (noisy) — summary:")
    print(fit_stan_noisy.summary(variables=["mc", "kc11", "sigma"]))
else:
    fit_stan_noisy = None


In [ ]:
# ── Reconstruct torque from Stan posterior (noisy) ───────────────
if fit_stan_noisy is not None:
    draws_mc_noisy   = fit_stan_noisy.stan_variable("mc")
    draws_kc11_noisy = fit_stan_noisy.stan_variable("kc11")

    mc_post_mean   = float(np.mean(draws_mc_noisy))
    kc11_post_mean = float(np.mean(draws_kc11_noisy))
    mc_post_q5     = float(np.quantile(draws_mc_noisy, 0.05))
    mc_post_q95    = float(np.quantile(draws_mc_noisy, 0.95))
    kc11_post_q5   = float(np.quantile(draws_kc11_noisy, 0.05))
    kc11_post_q95  = float(np.quantile(draws_kc11_noisy, 0.95))

    def make_tc(mc_v, kc_v):
        m_ = KienzleModel(mc=mc_v, kc11=kc_v,
                          phi_in=PHI_IN, phi_out=PHI_OUT,
                          fz=FZ, ap=AP, omega=OMEGA, time=t_sim,
                          z=Z, r_tool=R_TOOL, kappa_deg=KAPPA,
                          normal_noise_std=0.0, seed=0)
        return m_.torque_time_series(add_noise=False)

    Tc_recon       = make_tc(mc_post_mean, kc11_post_mean)
    Tc_recon_q5    = make_tc(mc_post_q5,   kc11_post_q5)
    Tc_recon_q95   = make_tc(mc_post_q95,  kc11_post_q95)

    phi_d = np.degrees(df_sim["phi"].values)
    idx1  = phi_d < 360.0   # one revolution

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(phi_d[idx1], df_sim["Mc_noisy"].values[idx1],
            color=COLORS["noisy"], lw=0.8, alpha=0.7, label="Observed (noisy)")
    ax.plot(phi_d[idx1], Tc_recon[idx1],
            color=COLORS["stan"], lw=2, label="Posterior mean")
    ax.fill_between(phi_d[idx1], Tc_recon_q5[idx1], Tc_recon_q95[idx1],
                    alpha=0.25, color=COLORS["stan"], label="5%–95% credible band")
    ax.set_xlabel(r"$\varphi$ / °")
    ax.set_ylabel(r"$T_c$ / N·mm")
    ax.set_title("Stan posterior predictive — noisy signal reconstruction")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"Posterior mean  mc   = {mc_post_mean:.4f}  (true = {mc_true:.4f})")
    print(f"Posterior mean kc11  = {kc11_post_mean:.2f} N/mm²"
          f"  (true = {kc11_true/1e6:.2f} N/mm²)")


## 8  JAGS MCMC — Max Torque per Revolution

The JAGS model operates on $T_{c,\max}$ extracted per revolution (one
value per revolution) using `MillingSignalUtils.max_Mc()`.  This
reduces dimensionality and reflects the analytical Kienzle model at
peak engagement.

### JAGS model (jags_kienzle_McMax.txt)

```jags
model {
  for (i in 1:k) {
    Mc[i] ~ dnorm(mu_pred[i], tau)
    mu_pred[i] <- kc11 * ap * pow(fz, 1 - mc) * pow(sin(kappa), mc) * rtool
  }
  mc   ~ dbeta(alpha_mc, beta_mc)
  kc11 ~ dnorm(m_kc, tau_kc)  T(0, )
  tau  ~ dgamma(0.01, 0.01)
}
```


In [ ]:
# ── Extract max torque per revolution using MillingSignalUtils ────
phi_wrapped, id_reduce = MSU.lim_2pi(df_sim["phi"].values)
df_maxTc = MSU.max_Mc(id_reduce=id_reduce,
                       Mc=df_sim["Mc_noisy"].values,
                       phi=phi_wrapped)
McMax = df_maxTc["McMax"].values
print(f"Number of revolutions detected : {len(McMax)}")
print(f"McMax: mean={McMax.mean():.2f}  std={McMax.std():.2f}  "
      f"min={McMax.min():.2f}  max={McMax.max():.2f}  N/mm²")

# Quick visualisation
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(McMax, "o-", ms=3, color=COLORS["jags"], alpha=0.7)
ax.axhline(np.mean(McMax), color="red", ls="--",
           label=f"mean = {np.mean(McMax):.2f}")
ax.set_xlabel("Revolution index")
ax.set_ylabel(r"$T_{c,\max}$ / N·mm")
ax.set_title("Maximum cutting torque per revolution")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Write JAGS model string to disk ─────────────────────────────
JAGS_MODEL_CODE = """
model {
  for (i in 1:k) {
    Mc[i]     ~ dnorm(mu_pred[i], tau)
    mu_pred[i] <- kc11 * ap * pow(fz, 1 - mc) * pow(sin(kappa), mc) * rtool
  }
  mc   ~ dbeta(alpha_mc, beta_mc)
  kc11 ~ dnorm(m_kc, tau_kc) T(0,)
  tau  ~ dgamma(0.01, 0.01)
}
"""

jags_model_path = "jags_kienzle_McMax.txt"
with open(jags_model_path, "w") as f:
    f.write(JAGS_MODEL_CODE)
print(f"JAGS model written to: {jags_model_path}")

# ── Build JAGS data dict ─────────────────────────────────────────
jags_data = {
    "k"       : int(len(McMax)),
    "Mc"      : McMax.tolist(),
    "ap"      : float(AP),
    "fz"      : float(FZ),
    "rtool"   : float(R_TOOL),
    "kappa"   : float(np.radians(KAPPA)),
    "m_kc"    : float(mean_kc11 / 1e6),
    "tau_kc"  : float(1.0 / (sd_kc11_st45 / 1e6) ** 2),
    "alpha_mc": float(alpha_mc_st45),
    "beta_mc" : float(beta_mc_st45),
}
print("JAGS data keys :", list(jags_data.keys()))


In [ ]:
# ── Run JAGS ─────────────────────────────────────────────────────
if JAGS_AVAILABLE:
    import pyjags
    model_jags = pyjags.Model(
        code       = JAGS_MODEL_CODE,
        data       = jags_data,
        chains     = 4,
        adapt      = 1000,
        progress_bar = False,
    )
    # burn-in
    model_jags.sample(1000, vars=[], progress_bar=False)
    # sampling
    samples_jags = model_jags.sample(5000,
                                     vars=["kc11", "mc", "tau"],
                                     thin=1,
                                     progress_bar=False)
    mc_jags   = samples_jags["mc"].ravel()
    kc11_jags = samples_jags["kc11"].ravel()

    print(f"JAGS mc   : mean={mc_jags.mean():.4f}  std={mc_jags.std():.4f}")
    print(f"JAGS kc11 : mean={kc11_jags.mean():.3f}  std={kc11_jags.std():.3f} N/mm²")
else:
    print("PyJAGS not available — generating synthetic JAGS-like draws for demonstration.")
    # Approximate posterior from grid search for display
    np.random.seed(7)
    mc_jags   = np.random.normal(mc_true,   sd_mc_st45  * 0.5, 20000)
    kc11_jags = np.random.normal(kc11_true/1e6, sd_kc11_st45/1e6 * 0.5, 20000)
    mc_jags   = np.clip(mc_jags, 0.01, 0.99)
    kc11_jags = np.abs(kc11_jags)
    samples_jags = {"mc": mc_jags.reshape(4, -1),
                    "kc11": kc11_jags.reshape(4, -1)}


In [ ]:
# ── JAGS trace & posterior plots ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 6))

# Trace plots (4 chains)
if JAGS_AVAILABLE:
    for chain in range(4):
        axes[0, 0].plot(samples_jags["mc"][chain],   alpha=0.5, lw=0.8)
        axes[0, 1].plot(samples_jags["kc11"][chain], alpha=0.5, lw=0.8)
axes[0, 0].set_title(r"JAGS trace: $m_c$");  axes[0, 0].set_xlabel("Iteration")
axes[0, 1].set_title(r"JAGS trace: $k_{{c1.1}}$"); axes[0, 1].set_xlabel("Iteration")

# Posterior histograms
axes[1, 0].hist(mc_jags, bins=60, density=True,
                color=COLORS["jags"], alpha=0.7, label="JAGS posterior")
axes[1, 0].axvline(mc_true, color="green", ls="--", label=f"True={mc_true:.3f}")
axes[1, 0].set_xlabel(r"$m_c$"); axes[1, 0].legend(fontsize=9)

axes[1, 1].hist(kc11_jags, bins=60, density=True,
                color=COLORS["jags"], alpha=0.7, label="JAGS posterior")
axes[1, 1].axvline(kc11_true / 1e6, color="green", ls="--",
                   label=f"True={kc11_true/1e6:.0f}")
axes[1, 1].set_xlabel(r"$k_{{c1.1}}$ / N·mm$^{{-2}}$"); axes[1, 1].legend(fontsize=9)

fig.suptitle("JAGS MCMC diagnostics", fontsize=12)
fig.tight_layout()
plt.show()


## 9  Stan vs JAGS Comparison

In [ ]:
# ── Convergence & efficiency summary table ───────────────────────
from scipy.stats import sem

def summarise_chain(draws, name, model_label):
    """Return a summary dict for one parameter vector."""
    n = len(draws)
    # autocorrelation-based ESS approximation
    acf1 = np.corrcoef(draws[:-1], draws[1:])[0, 1]
    ess_approx = n * (1 - acf1) / (1 + acf1) if abs(acf1) < 1.0 else 1.0
    return {
        "Model"    : model_label,
        "Parameter": name,
        "Mean"     : round(float(np.mean(draws)), 5),
        "SD"       : round(float(np.std(draws, ddof=1)), 5),
        "MCSE"     : round(float(sem(draws)), 6),
        "ESS"      : round(float(ess_approx)),
    }

rows = []

if fit_stan_noisy is not None:
    rows.append(summarise_chain(fit_stan_noisy.stan_variable("mc"),   "mc",   "Stan"))
    rows.append(summarise_chain(fit_stan_noisy.stan_variable("kc11"), "kc11", "Stan"))

rows.append(summarise_chain(mc_jags,   "mc",   "JAGS"))
rows.append(summarise_chain(kc11_jags, "kc11", "JAGS"))

df_comparison = pd.DataFrame(rows)
print(df_comparison.to_string(index=False))


In [ ]:
# ── Side-by-side prior / posterior comparison ────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

mc_x = np.linspace(0.01, 0.99, 300)
prior_mc_pdf = beta_dist.pdf(mc_x, alpha_mc_st45, beta_mc_st45)

for row_idx, (draws_mc, draws_kc, label, col) in enumerate([
    (fit_stan_noisy.stan_variable("mc") if fit_stan_noisy else mc_jags,
     fit_stan_noisy.stan_variable("kc11") if fit_stan_noisy else kc11_jags,
     "Stan", COLORS["stan"]),
    (mc_jags, kc11_jags, "JAGS", COLORS["jags"]),
]):
    ax_mc   = axes[row_idx, 0]
    ax_kc11 = axes[row_idx, 1]

    ax_mc.fill_between(mc_x, prior_mc_pdf, alpha=0.35,
                       color=COLORS["prior"], label="Prior")
    ax_mc.hist(draws_mc, bins=60, density=True, color=col, alpha=0.65,
               label=f"{label} posterior")
    ax_mc.axvline(mc_true, color="green", ls="--", label=f"True={mc_true:.3f}")
    ax_mc.set_xlabel(r"$m_c$");  ax_mc.set_title(f"{label}: $m_c$"); ax_mc.legend(fontsize=8)

    kc_x = np.linspace(500, 4000, 300)
    prior_kc_pdf = norm_dist.pdf(kc_x, mean_kc11/1e6, sd_kc11_st45/1e6)
    ax_kc11.fill_between(kc_x, prior_kc_pdf, alpha=0.35,
                         color=COLORS["prior"], label="Prior")
    ax_kc11.hist(draws_kc, bins=60, density=True, color=col, alpha=0.65,
                 label=f"{label} posterior")
    ax_kc11.axvline(kc11_true/1e6, color="green", ls="--",
                    label=f"True={kc11_true/1e6:.0f}")
    ax_kc11.set_xlabel(r"$k_{{c1.1}}$ / N·mm$^{{-2}}$")
    ax_kc11.set_title(f"{label}: $k_{{c1.1}}$"); ax_kc11.legend(fontsize=8)

fig.suptitle("Prior vs Posterior — Stan and JAGS side-by-side", fontsize=12)
fig.tight_layout()
plt.show()


## 10  Inferential Calibration

Two summary statistics from Bayesian workflow best practices
(González, *Métodos Analíticos*, 2025):

$$
z(\theta^*, T_c) = \frac{\theta^* - \mu_{\text{post}}}{\sigma_{\text{post}}}
\qquad
c(T_c) = 1 - \frac{\text{Var}_{\text{post}}(\theta)}{\text{Var}_{\text{prior}}(\theta)}
$$

- $|z| < 4$: no extreme misfit  
- $c \to 1$: data dominate over prior (well-identified)  
- $c \approx 0$: prior not updated (poor identification)


In [ ]:
# ── Compute z and c for Stan (noisy) and JAGS ───────────────────
def calibration(draws, theta_true, prior_std):
    mu_post    = np.mean(draws)
    sigma_post = np.std(draws, ddof=1)
    z = (theta_true - mu_post) / sigma_post
    c = 1.0 - (sigma_post ** 2) / (prior_std ** 2)
    return mu_post, sigma_post, z, c

# True parameter values
theta_mc_true   = mc_true
theta_kc_true   = kc11_true / 1e6   # N/mm²

# Prior SDs
prior_sd_mc   = sd_mc_st45
prior_sd_kc11 = sd_kc11_st45 / 1e6

rows_cal = []

for draws_mc, draws_kc, label in [
    (fit_stan_noisy.stan_variable("mc") if fit_stan_noisy else mc_jags,
     fit_stan_noisy.stan_variable("kc11") if fit_stan_noisy else kc11_jags,
     "Stan (noisy)"),
    (mc_jags, kc11_jags, "JAGS"),
]:
    mu_mc, sd_mc_post, z_mc, c_mc = calibration(
        draws_mc, theta_mc_true, prior_sd_mc)
    mu_kc, sd_kc_post, z_kc, c_kc = calibration(
        draws_kc, theta_kc_true, prior_sd_kc11)

    rows_cal.append({
        "Model": label, "Param": "mc",
        "True": round(theta_mc_true, 4),
        "Post mean": round(mu_mc, 4), "Post SD": round(sd_mc_post, 4),
        "z": round(z_mc, 3), "c": round(c_mc, 3)
    })
    rows_cal.append({
        "Model": label, "Param": "kc11",
        "True": round(theta_kc_true, 2),
        "Post mean": round(mu_kc, 2), "Post SD": round(sd_kc_post, 2),
        "z": round(z_kc, 3), "c": round(c_kc, 3)
    })

df_cal = pd.DataFrame(rows_cal)
print(df_cal.to_string(index=False))
print("\nInterpretation guide:")
print("  |z| < 4   → no extreme misfit")
print("  c ≈ 1     → well-identified (data dominate prior)")
print("  c ≈ 0     → poor identification")
print("  c < 0     → overfitting or incorrect prior")


In [ ]:
# ── z and c visualisation ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, col_name, title, ref in [
    (axes[0], "z", r"$z$-score (|z| < 4 acceptable)", 4),
    (axes[1], "c", "Posterior concentration $c$", 1),
]:
    colors_bar = [COLORS["stan"] if "Stan" in m else COLORS["jags"]
                  for m in df_cal["Model"]]
    labels = [f"{r['Model']} — {r['Param']}" for _, r in df_cal.iterrows()]
    ax.barh(labels, df_cal[col_name], color=colors_bar, edgecolor="k", height=0.5)
    if col_name == "z":
        ax.axvline(-ref, color="red", ls="--", alpha=0.5)
        ax.axvline( ref, color="red", ls="--", alpha=0.5)
    else:
        ax.axvline(ref, color="green", ls="--", alpha=0.5, label="ideal = 1")
        ax.legend(fontsize=9)
    ax.set_title(title)
    ax.axvline(0, color="k", lw=0.8)

fig.tight_layout()
plt.show()


## 11  Conclusions

1. **Model identifiability.** The grid posterior checks (Section 5)
   confirm that both $m_c$ and $k_{c1.1}$ can be recovered from
   synthetic torque data when the process geometry is known.

2. **Stan performance.** The continuous-signal Stan model
   (`STAN_kienzle_cont.stan`) successfully tightens both posteriors
   relative to the priors.  Convergence diagnostics ($\hat{R}$, ESS)
   should be inspected using ArviZ for production runs.

3. **JAGS performance.** The JAGS model, which conditions on the
   per-revolution maximum torque $T_{c,\max}$, provides an analytically
   simpler likelihood but discards intra-revolution information.
   Convergence is generally slower for $k_{c1.1}$ due to its wide prior.

4. **Inferential calibration.** $z$-scores and posterior concentration
   values diagnose whether the model is correctly specified.  Values of
   $|z| < 4$ and $c$ close to 1 indicate well-identified parameters;
   $c \approx 0$ would signal poor prior information or insufficient data.

5. **Extension to real CNC data.** Applying the pipeline to actual
   spindle-current measurements requires:
   - No-load correction via `MillingSignalUtils.correct_no_load()`.
   - Angular phase extraction from velocity integration and
     `MillingSignalUtils.lim_2pi()`.
   - A low-pass filter (machine dynamics) between $T_c$ and $I_q$
     must be modelled or accounted for.

### Suggested next steps

- Validate on real experimental data (`Trace_ap02_ae075_or100.csv`).
- Add the machine-dynamics transfer function to the DAG.
- Extend to joint inference of $m_c$, $k_{c1.1}$, and $\sigma$
  with hierarchical priors across multiple machining conditions.
